In [3]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# 1. Load dataset
data = load_breast_cancer()
X = data.data
y = data.target

# 2. Preprocessing
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Sub-split the training set for fast fitness evaluation during optimization
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# Dictionary to cache evaluated fitness values to avoid redundant training
fitness_cache = {}

# 3. Fitness function with caching and fast validation
def fitness(learning_rate):
    # Round to 6 decimal places to allow efficient caching of nearby values
    lr_key = round(float(learning_rate), 6)
    if lr_key in fitness_cache:
        return fitness_cache[lr_key]

    # Use a faster, lightweight model for evaluation
    model = MLPClassifier(
        hidden_layer_sizes=(16, 8),
        learning_rate_init=lr_key,
        max_iter=100,  # Reduced max_iter for evaluation speed
        random_state=42
    )

    model.fit(X_fit, y_fit)
    prediction = model.predict(X_val)
    error_rate = 1 - accuracy_score(y_val, prediction)

    fitness_cache[lr_key] = error_rate
    return error_rate

# 4. Initialize Cuckoo nests
n = 10
iterations = 15
pa = 0.25

# Seed for reproducibility
np.random.seed(42)
nests = np.random.uniform(0.001, 0.1, n)

# 5. Cuckoo Search
for i in range(iterations):
    for j in range(n):
        # Generate new solution using random walk
        step_size = np.random.normal(0, 0.01)
        new_nest = nests[j] + step_size
        new_nest = np.clip(new_nest, 0.001, 0.1)

        # Evaluate and update if the new solution is better
        if fitness(new_nest) < fitness(nests[j]):
            nests[j] = new_nest

    # Abandon poor nests with probability pa
    for j in range(n):
        if np.random.rand() < pa:
            nests[j] = np.random.uniform(0.001, 0.1)

    best_accuracy = (1 - min(fitness(x) for x in nests)) * 100
    print(f"Iteration: {i + 1:02d} | Best Validation Accuracy: {best_accuracy:.2f}%")

# 6. Find best solution
best_lr = min(nests, key=fitness)
print(f"\nOptimal Learning Rate: {best_lr:.6f}")

# 7. Train final AI model using the entire train set
final_model = MLPClassifier(
    hidden_layer_sizes=(16, 8),
    learning_rate_init=best_lr,
    max_iter=300,
    random_state=42
)
final_model.fit(X_train, y_train)

# 8. Final prediction
y_pred = final_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nFinal Classification Accuracy on Test Set: {accuracy * 100:.2f}%")

Iteration: 1 Best Accuracy: 98.24561403508771
Iteration: 2 Best Accuracy: 98.24561403508771
Iteration: 3 Best Accuracy: 98.24561403508771
Iteration: 4 Best Accuracy: 98.24561403508771
Iteration: 5 Best Accuracy: 98.24561403508771
Iteration: 6 Best Accuracy: 98.24561403508771


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


Iteration: 7 Best Accuracy: 98.24561403508771


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


Iteration: 8 Best Accuracy: 98.24561403508771


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


Iteration: 9 Best Accuracy: 98.24561403508771
Iteration: 10 Best Accuracy: 98.24561403508771
Iteration: 11 Best Accuracy: 98.24561403508771
Iteration: 12 Best Accuracy: 98.24561403508771
Iteration: 13 Best Accuracy: 98.24561403508771
Iteration: 14 Best Accuracy: 98.24561403508771
Iteration: 15 Best Accuracy: 98.24561403508771
Iteration: 16 Best Accuracy: 98.24561403508771
Iteration: 17 Best Accuracy: 98.24561403508771
Iteration: 18 Best Accuracy: 98.24561403508771
Iteration: 19 Best Accuracy: 98.24561403508771
Iteration: 20 Best Accuracy: 98.24561403508771

Optimal Learning Rate: 0.09307894470046413

Final Classification Accuracy: 98.24561403508771 %
